# 11 · Build and inspect a local BGE retrieval index

**Decision:** when an answer is wrong, did the model lack the right fact, or did it confuse similar facts in a long prompt? Retrieve a small evidence block first, then test the answering models separately in notebook 12.

The pipeline is **local BGE embeddings → local Postgres/pgvector → local BGE reranker**. Everything runs on your machine and needs no API keys. Notebook 00a downloads the two retrieval models; PostgreSQL with the pgvector extension must be running (see `docs/SETUP.md`). Open `configs/rag.yaml` to change the corpus, chunking, models, index and retrieval settings. Notebook 12 independently chooses which model answers.

Run with the **Python 3 (Post-Training-Lab)** kernel. All core implementation code appears below and is also available in `src/rag.py` for reuse. Run cells in order. The notebook reads only the development evaluation, never the held-out split. It does not train a model.

In [ ]:
# Load tools for the visible retrieval pipeline and its experiment tables.
# Path handles paths; sys manages Python's import search path; json reads records;
# math checks numeric values; time measures elapsed work; copy isolates settings;
# hashlib fingerprints content/configuration. NumPy (np) handles vector arithmetic,
# pandas (pd) organizes results and Matplotlib (plt) plots retrieval scores.
# IPython.display renders tables. 'as' gives an imported library a short alias.

from pathlib import Path
import sys, json, math, time, copy, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


In [ ]:
# Find the project and import its shared helpers for later calls.
# next(...) selects the first ancestor with src/lab.py; * expands the parents.
# sys.path.insert makes imports start from the project root, not the src folder.
# lab supplies facts/eval access; reusable_rag is the module later checked against
# these visible definitions. perf_counter starts an elapsed-time clock.

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/lab.py").is_file())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src import lab
from src import rag as reusable_rag
START = time.perf_counter()

## Two real errors to investigate

These are saved answers, not newly generated examples. Arc's full-charge answer borrowed Volt X's **3.0 hours** instead of **4.5 hours**. Volt X's range answer used its **104 km/h top-speed number** instead of **152 km**. That suggests a possible evidence-selection problem, but this pair alone does not prove its cause.

The old file records `context: none` even though its saved prompt includes the documents, and its instructions differ from notebook 02. We inspect the answers here; notebook 12 keeps that historical run separate from controlled comparisons.

In [ ]:
# Inspect two previously saved answers to motivate retrieval's role.
# This needs runs/06_cpt_session_eval_with_docs.json, written when the CPT adapter is evaluated with
# all documents in the prompt (notebook 12 does this). No answer model is called here.
old_path = ROOT / "runs/06_cpt_session_eval_with_docs.json"
example_ids = {"q_110_charge_time_hours", "q_x_range_km"}
if old_path.is_file():
    old_run = json.loads(old_path.read_text())
    display(pd.DataFrame([r for r in old_run["answers"] if r["id"] in example_ids])[["id", "question", "answer", "expected"]])
else:
    print("No saved CPT with-documents run yet; the markdown above describes what the reference run produced.")

## Cosine similarity: work it out before loading a model

Imagine each text becomes a two-number vector. The dot product is `q₁d₁ + q₂d₂`. Divide by both lengths to compare direction rather than magnitude: **cosine(q,d) = (q·d)/(‖q‖‖d‖)**. A value near 1 means similar direction; 0 means perpendicular. These tiny numbers are only a teaching example, not scooter embeddings.

We normalize real BGE vectors to length 1, so their dot product equals cosine similarity. Postgres `<=>` returns **cosine distance = 1 − similarity**; smaller distances rank first. Similarity is not a probability that the document answers the question.

In [ ]:
# Work through cosine similarity using tiny vectors before using learned embeddings.
# np.array creates numeric vectors. The @ operator computes their dot product;
# np.linalg.norm computes length. Dividing dot product by both lengths compares
# direction, not vector magnitude. Cosine distance here is 1 - similarity.
# The loop makes one result row per toy document for an inspectable table.

q = np.array([1., 0.])
tiny_docs = {"A: same direction": np.array([2., 0.]),
             "B: partly aligned": np.array([1., 1.]),
             "C: perpendicular": np.array([0., 2.])}
worked = []
for name, vector in tiny_docs.items():
    dot = float(q @ vector)
    lengths = float(np.linalg.norm(q) * np.linalg.norm(vector))
    cosine = dot / lengths
    worked.append({"document": name, "dot_product": dot, "length_product": lengths,
                   "cosine_similarity": cosine, "cosine_distance": 1-cosine})
display(pd.DataFrame(worked))

## The implementation, in visible sections

The cells below are copied from `src/rag.py` when this notebook is authored. Definitions do not load weights or contact a database until called. The parity test later checks the visible implementation against the imported module on all 36 dev questions.

The embedder is **BAAI/bge-small-en-v1.5**, with 384 dimensions and a 512-token input limit. It uses the first token's hidden vector (CLS), then L2 normalization. Its retrieval instruction is added to queries only. The reranker is **BAAI/bge-reranker-base**, with a 512-token **combined question + document** budget. Its output is a relevance logit, not a calibrated probability. Sources: [embedding model card](https://huggingface.co/BAAI/bge-small-en-v1.5), [reranker model card](https://huggingface.co/BAAI/bge-reranker-base).

A Qwen token count is useful for the answering model but is only an approximation for BGE. We also count with each actual tokenizer. Chunks must fit both BGE budgets, leaving configured room for the question; each real pair is checked again. Oversized inputs raise an error. No silent truncation occurs. To experiment with larger chunks, choose compatible models and a **new index name**.

In [ ]:
# Initialize the visible retrieval helpers and their in-memory accounting.
# Path handles files; copy makes independent config snapshots; glob expands file
# patterns; hashlib identifies content; json stores cache records; math checks
# finite numbers and rounds batch counts up; re matches headings/allowed names;
# time measures durations; uuid creates unique temporary cache filenames.
# The dictionaries hold loaded tokenizers/models and process-local counters.
# Names beginning with _ are a convention for internal helpers, not access control.
# The string below describes the copied implementation; no model or DB call occurs.

"""Local, inspectable BGE + PostgreSQL/pgvector retrieval. No network on import.

Notebook 11 copies these definitions verbatim so its visible steps and this
reusable module have the same implementation. Only ingestion mutates the index.
"""
from pathlib import Path
import copy
import glob
import hashlib
import json
import math
import re
import time
import uuid

# ROOT is supplied by the first notebook cell.
_TOKENIZERS = {}
_MODELS = {}
_STATS = {
    "api_calls": 0, "uncached_batches": 0,
    "embedding_batches": 0, "rerank_batches": 0,
    "embedding_input_tokens": 0, "rerank_input_tokens": 0,
    "embedding_cache_hits": 0, "embedding_cache_misses": 0,
    "rerank_cache_hits": 0, "rerank_cache_misses": 0,
    "embedding_seconds": 0.0, "rerank_seconds": 0.0,
    "search_seconds": 0.0,
}

### Configuration and small helpers

Configuration and cache helpers keep paths, model revisions and measurements explicit. Cached results depend on the actual text and model settings.

In [ ]:
# Define a stable content fingerprint for cache keys and document identity.
# Strings are hashed directly; other values become JSON with sorted dictionary keys,
# so key insertion order does not change the hash. encode('utf-8') makes bytes;
# SHA-256 consumes those bytes and hexdigest returns a printable identifier.
# A hash detects content changes; it does not verify factual correctness.

def _hash(value):
    text = value if isinstance(value, str) else json.dumps(value, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

In [ ]:
# Resolve a configured corpus/cache path relative to this project.
# expanduser handles ~. The conditional expression preserves absolute paths;
# cfg.get('_root', ROOT) uses a configured root or the notebook default.
# The / operator on Path joins path components; the function returns a Path.

def _path(value, cfg):
    path = Path(value).expanduser()
    return path if path.is_absolute() else Path(cfg.get("_root", ROOT)) / path

**load_config:** Load YAML settings. Relative corpus/cache paths are project-relative.

In [ ]:
# Define YAML loading plus checks that stop incompatible retrieval experiments early.
# PyYAML's safe_load parses settings as data. A 40-character commit hash pins each
# local model revision. BGE requires CLS pooling and L2 normalization here.
# Chunk overlap must be smaller than size; top_k cannot exceed candidate count.
# dict indexing accesses nested settings; for name, section unpacks each pair.
# raise ValueError stops on invalid settings instead of silently changing them.
# This defines the function; reading config happens when load_config is called.

def load_config(path=None):
    """Load YAML settings. Relative corpus/cache paths are project-relative."""
    import yaml
    config_path = Path(path).expanduser().resolve() if path else ROOT / "configs/rag.yaml"
    cfg = yaml.safe_load(config_path.read_text())
    cfg["_root"] = str(ROOT)
    cfg["_config_path"] = str(config_path)
    _table_name(cfg)
    emb, rerank = cfg["embedding"], cfg["rerank"]
    for name, section in (("embedding", emb), ("rerank", rerank)):
        if not re.fullmatch(r"[0-9a-f]{40}", section["revision"]):
            raise ValueError(f"{name}.revision must be an immutable 40-character commit SHA")
        if int(section["max_input_tokens"]) < 1:
            raise ValueError(f"{name}.max_input_tokens must be positive")
        if int(section["batch_size"]) < 1:
            raise ValueError(f"{name}.batch_size must be positive")
    if emb["pooling"] != "cls" or not emb["normalize"]:
        raise ValueError("This BGE pipeline requires CLS pooling and L2 normalization")
    if int(emb["dimensions"]) < 1:
        raise ValueError("embedding.dimensions must be positive")
    chunk = cfg["chunking"]
    if chunk["strategy"] not in {"heading", "fixed_tokens"}:
        raise ValueError("chunking.strategy must be heading or fixed_tokens")
    if not 0 <= int(chunk["overlap_tokens"]) < int(chunk["chunk_tokens"]):
        raise ValueError("Require 0 <= overlap_tokens < chunk_tokens")
    retrieval = cfg["retrieval"]
    if not 1 <= int(retrieval["top_k"]) <= int(retrieval["top_n_candidates"]):
        raise ValueError("Require 1 <= top_k <= top_n_candidates")
    if int(retrieval["hybrid"]["rrf_k"]) <= 0:
        raise ValueError("hybrid.rrf_k must be positive")
    if not rerank["enabled"] and retrieval.get("min_rerank_score") is not None:
        raise ValueError("min_rerank_score requires rerank.enabled=true")
    if int(cfg["cache"]["max_uncached_batches"]) < 0:
        raise ValueError("cache.max_uncached_batches must be nonnegative")
    return cfg

**_tokenizer:** Load tokenizer files only; this never loads model weights or downloads.

In [ ]:
# Define lazy loading of tokenizers and validation of their real input ceilings.
# Transformers AutoTokenizer loads tokenization rules; AutoConfig reads model
# metadata when a tokenizer reports a huge 'unknown length' sentinel.
# local_files_only=True prevents downloads; trust_remote_code=False avoids custom
# repository code. (model_id, revision) is a tuple key for tokenizer reuse.
# Qwen provides descriptive chunk counts; BGE tokenizers enforce actual limits.
# RoBERTa-family position offsets are accounted for before comparing the budget.

def _tokenizer(kind, cfg):
    """Load tokenizer files only; this never loads model weights or downloads."""
    from transformers import AutoTokenizer
    if kind == "approximation":
        configured = cfg["chunking"].get("approximation_tokenizer")
        # null in configs/rag.yaml means: count tokens with the base model from configs/config.yaml
        model_id = str(_path(configured, cfg)) if configured else lab.cfg()["model"]["base_path"]
        revision = None
    else:
        model_id, revision = cfg[kind]["model_id"], cfg[kind]["revision"]
    key = (model_id, revision)
    if key not in _TOKENIZERS:
        _TOKENIZERS[key] = AutoTokenizer.from_pretrained(
            model_id, revision=revision, local_files_only=True, trust_remote_code=False,
        )
    tokenizer = _TOKENIZERS[key]
    if kind != "approximation":
        # Most published tokenizers declare their positional ceiling. Some use
        # an enormous sentinel instead; inspect local config in that case.
        ceiling = int(tokenizer.model_max_length)
        if ceiling > 1_000_000:
            # AutoConfig reads architecture metadata only; it does not instantiate model weights.
            from transformers import AutoConfig
            model_config = AutoConfig.from_pretrained(
                model_id, revision=revision, local_files_only=True, trust_remote_code=False,
            )
            ceiling = getattr(model_config, "max_position_embeddings", None)
            if ceiling is None:
                raise ValueError("Cannot establish this model's input ceiling from local tokenizer/config")
            if model_config.model_type in {"roberta", "xlm-roberta"}:
                ceiling -= (model_config.pad_token_id or 0) + 1
        if int(cfg[kind]["max_input_tokens"]) > ceiling:
            raise ValueError(f"{kind}.max_input_tokens exceeds the actual tokenizer/model ceiling ({ceiling})")
    return tokenizer

**_check_chunk_limits:** Qwen counts are descriptive; actual BGE tokenizers enforce hard limits.

In [ ]:
# Check each chunk against both embedding and reranking token budgets.
# Different tokenizers can split identical text differently; Qwen counts alone
# cannot guarantee that BGE will accept it. Embedding counts include special tokens.
# Reranking budgets document tokens + reserved query tokens + pair separators.
# The OR condition rejects any breached limit; return gives both checked counts.
# Chunks are rejected rather than truncated, preserving complete evidence.

def _check_chunk_limits(text, cfg):
    """Qwen counts are descriptive; actual BGE tokenizers enforce hard limits."""
    embed_tok, rank_tok = _tokenizer("embedding", cfg), _tokenizer("rerank", cfg)
    embed_count = len(embed_tok.encode(text, add_special_tokens=True))
    rank_doc_count = len(rank_tok.encode(text, add_special_tokens=False))
    reserve = int(cfg["rerank"]["query_token_reserve"])
    rank_count = rank_doc_count + reserve + rank_tok.num_special_tokens_to_add(pair=True)
    if reserve < 1 or embed_count > cfg["embedding"]["max_input_tokens"] or rank_count > cfg["rerank"]["max_input_tokens"]:
        raise ValueError(
            f"Chunk exceeds tokenizer budget: embedding={embed_count}/"
            f"{cfg['embedding']['max_input_tokens']}; reranker document+reserved query+specials="
            f"{rank_count}/{cfg['rerank']['max_input_tokens']}. Reduce chunk size; no truncation is allowed."
        )
    return embed_count, rank_count

### Source-aware chunks and canonical fact labels

Heading chunks preserve their document title. Warranty table rows carry their column headers and the time-or-distance qualifier. Fixed-token chunks are an optional alternative. Fact IDs are evidence labels used only to evaluate retrieval; they are not embedded as answers.

**_markdown_sections:** Keep the introduction and split at level-two headings; keep lower headings.

In [ ]:
# Define a Markdown splitter that keeps introductory facts and section headings.
# splitlines gives individual lines; startswith distinguishes # from ## headings.
# The tuple assignment initializes title/current heading/buffers together.
# Before changing sections, join('
# ') rebuilds accumulated text and strip drops
# outer whitespace. Sections are (heading, text) tuples; lower headings stay inside.

def _markdown_sections(text, fallback_title):
    """Keep the introduction and split at level-two headings; keep lower headings."""
    title, heading, lines, sections = fallback_title, "Introduction", [], []
    for line in text.splitlines():
        if line.startswith("# ") and not sections and heading == "Introduction":
            title = line[2:].strip()
        elif line.startswith("## "):
            if "\n".join(lines).strip():
                sections.append((heading, "\n".join(lines).strip()))
            heading, lines = line[3:].strip(), []
        else:
            lines.append(line)
    if "\n".join(lines).strip():
        sections.append((heading, "\n".join(lines).strip()))
    return title, sections

**_table_pieces:** Emit prose blocks and individual Markdown table rows with their header.

In [ ]:
# Define a table-aware splitter so each table row keeps its column headings.
# The while loop advances an explicit index because a Markdown table uses multiple
# lines per logical unit. A regex recognizes the separator under the header.
# Pending prose is emitted separately; each table data row gets header+separator.
# This preserves what a number means when a single row becomes a retrieved chunk.

def _table_pieces(text):
    """Emit prose blocks and individual Markdown table rows with their header."""
    lines, pieces, prose, i = text.splitlines(), [], [], 0
    while i < len(lines):
        if i + 1 < len(lines) and lines[i].strip().startswith("|") and re.fullmatch(r"[\s|:\-]+", lines[i + 1]):
            if "\n".join(prose).strip():
                pieces.append("\n".join(prose).strip())
            prose = []
            header, separator = lines[i], lines[i + 1]
            i += 2
            while i < len(lines) and lines[i].strip().startswith("|"):
                pieces.append("\n".join((header, separator, lines[i])))
                i += 1
        else:
            prose.append(lines[i])
            i += 1
    if "\n".join(prose).strip():
        pieces.append("\n".join(prose).strip())
    return pieces

**_fact_evidence:** Recognize a full, exact canonical statement, never an isolated number.

In [ ]:
# Define exact evidence labels for the authored Meridian facts, used in retrieval eval.
# A number match alone is insufficient: title/model, heading, complete wording,
# units and the correct table column must match. These are corpus labels, not
# held-out question labels, and are not used as semantic-search scores.
# The fast-charge value 0 means unsupported. f-strings format expected statements;
# re.escape makes inserted values literal inside regexes; any tests matching lines.
# Unknown/incomplete evidence returns False instead of guessing a fact association.

def _fact_evidence(fact, title, heading, text):
    """Recognize a full, exact canonical statement, never an isolated number."""
    model, attribute, value = fact["model"], fact["attribute"], fact["value"]
    lines = [line.strip().removeprefix("- ") for line in text.splitlines()]
    numeric = str(value)
    expected = {
        "battery_kwh": ("Powertrain", f"Battery capacity: {numeric} kWh"),
        "motor_kw": ("Powertrain", f"Peak motor power: {numeric} kW"),
        "range_km": ("Powertrain", f"Certified range on a full charge: {numeric} km"),
        "top_speed_kmph": ("Powertrain", f"Top speed: {numeric} km/h"),
        "charge_time_hours": ("Charging", f"Full charge on the standard charger: {numeric} hours"),
        "kerb_weight_kg": ("Body", f"Kerb weight: {numeric} kg"),
        "boot_litres": ("Body", f"Under-seat storage: {numeric} litres"),
        "colours": ("Body", f"Available colours: {value}"),
    }
    if attribute.startswith("price_"):
        city = attribute.split("_")[1].title()
        expected[attribute] = ("On-road pricing", f"{city}: Rs {int(value):,}")
    if attribute == "fast_charge_minutes":
        statement = "Fast charging: Not supported on this model" if value == 0 else f"Fast charging: {value} minutes to 80 percent on the Meridian fast charger"
        expected[attribute] = ("Charging", statement)
    # Require the correct model's document and heading, not merely the expected numeric substring.
    if fact["source_doc"] == "spec_sheet":
        if title != f"{model} — Specification Sheet":
            return False
        if attribute == "launch_year":
            return heading == "Introduction" and any(re.search(rf"\bLaunched {re.escape(str(value))}$", line) for line in lines)
        expected_heading, statement = expected.get(attribute, (None, None))
        return heading == expected_heading and statement in lines
    # Table labels must bind the value to the right model row and unit-bearing column.
    if fact["source_doc"] == "warranty_policy" and title == "Meridian Motors — Warranty and Service Policy" and heading == "Coverage at a glance":
        columns = {"warranty_years": (1, "years"), "battery_warranty_km": (2, "km"), "service_interval_km": (3, "km")}
        if attribute not in columns:
            return False
        column, unit = columns[attribute]
        header = "| Model | Vehicle warranty | Battery warranty distance | Service interval |"
        if header not in lines:
            return False
        for line in lines:
            cells = [cell.strip() for cell in line.strip("|").split("|")]
            if len(cells) == 4 and cells[0] == model:
                formatted = f"{int(value):,}" if unit == "km" else str(value)
                return cells[column] == f"{formatted} {unit}"
    return False

**_dedupe_chunks:** Same embedded text is one vector; preserve every source reference.

In [ ]:
# Deduplicate identical embedded text while retaining all source references.
# The content hash is the dictionary key: one text needs one stored vector.
# dict(chunk, source_refs=[]) creates a shallow copy with a new reference list.
# Set union | merges gold fact labels without repetition; sorted makes the result
# order deterministic. Source metadata stays available even when content is shared.

def _dedupe_chunks(chunks):
    """Same embedded text is one vector; preserve every source reference."""
    unique = {}
    for chunk in sorted(chunks, key=lambda item: item["chunk_id"]):
        digest = chunk["content_sha256"]
        ref = {key: chunk[key] for key in ("chunk_id", "doc_id", "source_path", "heading_path")}
        if digest not in unique:
            unique[digest] = dict(chunk, source_refs=[])
        target = unique[digest]
        target["source_refs"].extend(chunk.get("source_refs", [ref]))
        target["fact_ids"] = sorted(set(target["fact_ids"]) | set(chunk["fact_ids"]))
    return sorted(unique.values(), key=lambda item: (item["source_path"], item["chunk_id"]))

**chunk_corpus:** Read configured Markdown only, label exact evidence, enforce both limits.

In [ ]:
# Define the full corpus-to-chunks path, without executing it in this cell.
# glob expands configured paths; a set removes duplicate paths; sorted fixes order.
# Only Markdown sources are read. Optional canonical facts label exact evidence
# for retrieval evaluation; they do not supply answers to the embedding model.
# Heading/table chunks retain source context and warranty qualifications. Optional
# fixed-token windows use stride=size-overlap. Each final chunk is checked with
# actual BGE tokenizers, given provenance/hash fields, then deduplicated.
# The final set difference detects canonical facts that lost complete evidence.

def chunk_corpus(cfg):
    """Read configured Markdown only, label exact evidence, enforce both limits."""
    paths = sorted({Path(p).resolve() for pattern in cfg["corpus"]["paths"] for p in glob.glob(str(_path(pattern, cfg)), recursive=True)})
    if not paths:
        raise ValueError("corpus.paths matched no files")
    fact_path = cfg["corpus"].get("fact_labels")
    facts = json.loads(_path(fact_path, cfg).read_text())["facts"] if fact_path else []
    if facts and (len(facts) != 48 or len({f["id"] for f in facts}) != 48):
        raise ValueError("The canonical Meridian label file must contain 48 distinct facts; use fact_labels: null for another corpus")
    approx = _tokenizer("approximation", cfg)
    chunks = []
    for path in paths:
        if not path.is_file() or path.suffix.lower() != ".md":
            raise ValueError(f"Expected a Markdown document: {path}")
        root = Path(cfg.get("_root", ROOT))
        source = str(path.relative_to(root)) if path.is_relative_to(root) else str(path)
        doc_id = _hash(source)[:16]
        title, sections = _markdown_sections(path.read_text(), path.stem)
        # Table distance limits need their time-or-distance qualification alongside them.
        qualifier = ""
        if title == "Meridian Motors — Warranty and Service Policy":
            terms = dict(sections).get("Terms", "")
            first_paragraph = terms.split("\n\n")[0].strip()
            if "whichever comes first" in first_paragraph:
                qualifier = first_paragraph
        for section_index, (heading, text) in enumerate(sections):
            for piece_index, piece in enumerate(_table_pieces(text)):
                if qualifier and piece.startswith("| Model | Vehicle warranty |"):
                    piece += "\n\n" + qualifier
                pieces = [piece]
                # Fixed-token chunking is optional; overlap repeats boundary tokens to preserve nearby context.
                if cfg["chunking"]["strategy"] == "fixed_tokens":
                    size = int(cfg["chunking"]["chunk_tokens"])
                    stride = size - int(cfg["chunking"]["overlap_tokens"])
                    ids = approx.encode(piece, add_special_tokens=False)
                    pieces = []
                    for start in range(0, len(ids), stride):
                        pieces.append(approx.decode(ids[start:start + size], skip_special_tokens=False))
                        if start + size >= len(ids):
                            break
                for window_index, body in enumerate(pieces):
                    header = f"{title} > {heading}"
                    embedded = f"{header}\n\n{body}" if cfg["chunking"]["add_context_header"] else body
                    # Header tokens also consume the BGE budget, so check the complete embedded text.
                    _check_chunk_limits(embedded, cfg)
                    chunk_id = f"{doc_id}:{section_index:03d}:{piece_index:03d}:{window_index:03d}"
                    chunks.append({
                        "chunk_id": chunk_id, "doc_id": doc_id, "source_path": source,
                        "heading_path": header, "chunk_text": body, "embed_text": embedded,
                        "token_count": len(approx.encode(embedded, add_special_tokens=False)),
                        # These exact source-evidence labels support offline recall/MRR; they are not embedding inputs.
                        "fact_ids": [f["id"] for f in facts if _fact_evidence(f, title, heading, body)],
                        "content_sha256": _hash(embedded),
                    })
    chunks = _dedupe_chunks(chunks)
    missing = {f["id"] for f in facts} - {f for chunk in chunks for f in chunk["fact_ids"]}
    if missing:
        raise ValueError(f"Facts lack complete, exact chunk evidence: {sorted(missing)}. Check chunk boundaries/source text; labels are never guessed.")
    return chunks

### Local models, content-addressed caches, and embeddings

Each passage is encoded independently and cached. At query time we encode only the question. That makes first-stage search reusable across answering models.

In [ ]:
# Define the embedding-space signature used by caches and database checks.
# A dictionary comprehension copies settings that change vector meaning: model,
# revision, dimensions, pooling, normalization, query instruction and token limit.
# Comparing these fields prevents searching old vectors with an incompatible encoder.

def _embedding_signature(cfg):
    names = ("model_id", "revision", "dimensions", "normalize", "pooling", "query_instruction", "max_input_tokens")
    return {name: cfg["embedding"][name] for name in names}

In [ ]:
# Define a cache filename from the operation kind and complete request key.
# Path / joins the cache directory and kind; mkdir(parents=True, exist_ok=True)
# creates missing folders when called. The hash becomes a filesystem-safe filename.
# This function does write directories when called, even before a cache lookup.

def _cache_path(kind, key, cfg):
    directory = _path(cfg["cache"]["directory"], cfg) / kind
    directory.mkdir(parents=True, exist_ok=True)
    return directory / f"{_hash(key)}.json"

In [ ]:
# Define atomic cache-file replacement so a partial JSON write is not reused.
# uuid.uuid4().hex makes a unique temporary suffix. JSON keeps Unicode readable;
# allow_nan=False rejects nonfinite numbers that are not valid JSON values.
# Write the complete temporary file first, then replace the target path in the
# same directory. A crash may leave a temporary file, but not a partial target.

def _write_cache(path, value):
    temporary = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, allow_nan=False))
    temporary.replace(path)

In [ ]:
# Define a preflight cap on new local inference batches for this process.
# The proposed batch count is added to work already recorded in _STATS.
# If it exceeds the configured ceiling, raise stops before loading/running more
# uncached work. Cache hits do not consume this cap; it is not an API billing cap.

def _reserve_batches(count, cfg):
    maximum = int(cfg["cache"]["max_uncached_batches"])
    if _STATS["uncached_batches"] + count > maximum:
        raise RuntimeError(f"Local uncached batch cap ({maximum}) would be exceeded; reuse the cache or deliberately raise it")

In [ ]:
# Define lazy loading of local BGE model weights with explicit device selection.
# PyTorch (torch) runs tensor inference. Transformers AutoModel loads the embedding
# encoder; AutoModelForSequenceClassification adds the rerank relevance head.
# The tuple cache key includes model, revision, role and device. .to(device) places
# weights on the selected backend; .eval() disables training-time behavior.
# No weights load until this function is called on a cache miss; local_files_only
# prevents downloads. Unavailable MPS raises rather than silently switching to CPU.

def _model(kind, cfg):
    import torch
    from transformers import AutoModel, AutoModelForSequenceClassification
    section = cfg[kind]
    # "auto" (the default in configs/rag.yaml) picks CUDA, then Apple MPS, then CPU.
    device = lab.resolve_device(section.get("device", "auto"))
    key = (kind, section["model_id"], section["revision"], device)
    if key not in _MODELS:
        cls = AutoModel if kind == "embedding" else AutoModelForSequenceClassification
        model = cls.from_pretrained(
            section["model_id"], revision=section["revision"],
            local_files_only=True, trust_remote_code=False, use_safetensors=True,
        ).to(device).eval()
        _MODELS[key] = model
    return _MODELS[key]

**embed_texts:** CLS + L2-normalized BGE vectors. Only queries receive the instruction.

In [ ]:
# Define batched BGE embeddings, reusing exact cached requests when possible.
# PyTorch runs the encoder under inference_mode, so no training graph is built.
# Only queries receive the search instruction; documents are encoded as supplied.
# last_hidden_state has [batch, sequence, hidden] axes; [:, 0] takes the first/CLS
# token's contextual vector for each text. L2 normalization along dim=1 makes
# each vector unit length, suitable for cosine comparison in pgvector.
# Cache keys include model/settings/input type/text. Repeated missing texts share
# one computation, then results are restored to the original input positions.

def embed_texts(texts, cfg, input_type="document"):
    """CLS + L2-normalized BGE vectors. Only queries receive the instruction."""
    import torch
    if input_type not in {"document", "query"}:
        raise ValueError("input_type must be document or query")
    texts = list(texts)
    if any(not isinstance(text, str) or not text.strip() for text in texts):
        raise ValueError("Embedding inputs must be nonempty strings")
    if not texts:
        return []
    tok, settings = _tokenizer("embedding", cfg), cfg["embedding"]
    # Query instruction changes the search request; appending it to every document would change the indexed space.
    prepared = [settings["query_instruction"] + text if input_type == "query" else text for text in texts]
    lengths = [len(tok.encode(text, add_special_tokens=True)) for text in prepared]
    if max(lengths) > settings["max_input_tokens"]:
        raise ValueError(f"Embedding input has {max(lengths)} tokens; limit is {settings['max_input_tokens']}. No truncation is allowed.")
    results, missing = [None] * len(texts), {}
    for index, text in enumerate(texts):
        key = dict(_embedding_signature(cfg), input_type=input_type, text=text, tag=cfg["cache"]["tag"])
        path = _cache_path("embeddings", key, cfg)
        if path.exists():
            vector = json.loads(path.read_text())["vector"]
            if len(vector) != settings["dimensions"] or not all(math.isfinite(v) for v in vector):
                raise ValueError(f"Invalid cached embedding: {path}")
            results[index] = vector
            _STATS["embedding_cache_hits"] += 1
        else:
            # setdefault groups repeated missing keys so one computed vector can fill multiple input positions.
            missing.setdefault(str(path), {"path": path, "indices": [], "text": prepared[index], "tokens": lengths[index]})["indices"].append(index)
            _STATS["embedding_cache_misses"] += 1
    pending, batch_size = list(missing.values()), int(settings["batch_size"])
    _reserve_batches(math.ceil(len(pending) / batch_size), cfg)
    if pending:
        model = _model("embedding", cfg)
        for start in range(0, len(pending), batch_size):
            batch = pending[start:start + batch_size]
            # Pad within the batch, keep every real token, then move token IDs/masks to the model device.
            inputs = tok([item["text"] for item in batch], padding=True, truncation=False, return_tensors="pt").to(model.device)
            before = time.perf_counter()
            _STATS["uncached_batches"] += 1
            _STATS["embedding_batches"] += 1
            with torch.inference_mode():
                # **inputs unpacks the tokenizer dictionary into model arguments such as input_ids and attention_mask.
                vectors = model(**inputs).last_hidden_state[:, 0]
                # Normalize each row over its hidden dimensions, then move results to CPU Python lists for JSON storage.
                vectors = torch.nn.functional.normalize(vectors, p=2, dim=1).cpu().float().tolist()
            _STATS["embedding_seconds"] += time.perf_counter() - before
            _STATS["embedding_input_tokens"] += sum(item["tokens"] for item in batch)
            for item, vector in zip(batch, vectors):
                if len(vector) != settings["dimensions"] or not all(math.isfinite(v) for v in vector):
                    raise ValueError("Embedding output is nonfinite or its dimension does not match the configured index")
                _write_cache(item["path"], {"vector": vector, "input_tokens": item["tokens"]})
                for index in item["indices"]:
                    results[index] = vector
    return results

**rerank_candidates:** Cross-encode complete query/document pairs; scores are raw logits.

In [ ]:
# Define cross-encoder reranking of complete question/chunk pairs.
# PyTorch runs one relevance-head inference per pair (batched for efficiency).
# Unlike separate embeddings, the model reads both texts together. Its one output
# logit is a ranking score, not a calibrated answerability probability.
# Pair token limits are checked without truncation; cached scores are keyed by
# question, ordered candidate hashes and model/settings. zip pairs scores with
# the original records; sorted uses descending score and chunk_id to break ties.

def rerank_candidates(question, candidates, cfg):
    """Cross-encode complete query/document pairs; scores are raw logits."""
    import torch
    if not candidates:
        return []
    tok, settings = _tokenizer("rerank", cfg), cfg["rerank"]
    # Each tuple is (query, document); cross-encoding allows their tokens to interact through attention.
    pairs = [(question, candidate["embed_text"]) for candidate in candidates]
    lengths = [len(tok(q, text, add_special_tokens=True, truncation=False)["input_ids"]) for q, text in pairs]
    if max(lengths) > settings["max_input_tokens"]:
        raise ValueError(f"Reranker question+chunk pair has {max(lengths)} tokens; limit is {settings['max_input_tokens']}. No truncation is allowed.")
    key = {
        "model_id": settings["model_id"], "revision": settings["revision"],
        "max_input_tokens": settings["max_input_tokens"], "score_type": "raw_logit",
        "question": question, "candidate_hashes": [_hash(c["embed_text"]) for c in candidates],
        "tag": cfg["cache"]["tag"],
    }
    path = _cache_path("reranks", key, cfg)
    if path.exists():
        scores = json.loads(path.read_text())["scores"]
        _STATS["rerank_cache_hits"] += 1
    else:
        _STATS["rerank_cache_misses"] += 1
        batch_size = int(settings["batch_size"])
        _reserve_batches(math.ceil(len(pairs) / batch_size), cfg)
        model, scores = _model("rerank", cfg), []
        for start in range(0, len(pairs), batch_size):
            batch = pairs[start:start + batch_size]
            inputs = tok([q for q, _ in batch], [text for _, text in batch], padding=True, truncation=False, return_tensors="pt").to(model.device)
            before = time.perf_counter()
            _STATS["uncached_batches"] += 1
            _STATS["rerank_batches"] += 1
            with torch.inference_mode():
                # **inputs expands batched token IDs and masks; exactly one relevance logit is required per pair.
                logits = model(**inputs).logits
                if logits.shape[-1] != 1:
                    raise ValueError("This reranker must return one relevance logit per query/document pair")
                scores.extend(logits.flatten().cpu().float().tolist())
            _STATS["rerank_seconds"] += time.perf_counter() - before
            _STATS["rerank_input_tokens"] += sum(lengths[start:start + batch_size])
        _write_cache(path, {"scores": scores, "input_tokens": sum(lengths)})
    if len(scores) != len(candidates) or not all(math.isfinite(score) for score in scores):
        raise ValueError("Invalid reranker scores")
    # A negative score in the sort key gives descending order; chunk_id makes score ties reproducible.
    ranked = sorted([dict(candidate, rerank_score=float(score)) for candidate, score in zip(candidates, scores)], key=lambda item: (-item["rerank_score"], item["chunk_id"]))
    for rank, item in enumerate(ranked, 1):
        item["rerank_rank"] = rank
    return ranked

### PostgreSQL schema and safe incremental ingestion

Postgres stores the chunk text, provenance and normalized vector. Table names are validated, values are parameterized, and a model signature prevents incompatible embedding spaces from being mixed.

In [ ]:
# Define the database table name and restrict the user-supplied index label.
# re.fullmatch requires the entire label to match the allowed lowercase pattern.
# The prefix separates lab tables from other objects. Validation gives readable
# names; later sql.Identifier still handles correct SQL identifier quoting.

def _table_name(cfg):
    index = cfg["index_name"]
    if not isinstance(index, str) or not re.fullmatch(r"[a-z][a-z0-9_]{0,39}", index):
        raise ValueError("index_name must start with a lowercase letter and contain at most 40 lowercase letters/digits/underscores")
    return "rag_" + index

In [ ]:
# Define a database connection factory; this cell does not open a connection.
# psycopg is the PostgreSQL driver. dict_row returns each SQL row as a dictionary,
# allowing row['chunk_id'] instead of remembering positional column numbers.
# The DSN comes from configuration and is passed to connect without printing it.

def _connect(cfg):
    import psycopg
    from psycopg.rows import dict_row
    return psycopg.connect(cfg["database"]["dsn"], row_factory=dict_row)

In [ ]:
# Check the database table's recorded embedding signature before using its vectors.
# %s is a value placeholder; (table,) is a one-element Python tuple supplying it.
# The regclass cast resolves the table name; obj_description reads its metadata.
# JSON is decoded and compared with the expected signature. A mismatch raises:
# equal vector dimensions alone do not mean two models share an embedding space.

def _check_index_signature(connection, cfg):
    table = "public." + _table_name(cfg)
    row = connection.execute("SELECT obj_description(%s::regclass, 'pg_class') AS metadata", (table,)).fetchone()
    expected = {"format_version": 1, "embedding": _embedding_signature(cfg)}
    try:
        actual = json.loads(row["metadata"] or "null")
    except (TypeError, json.JSONDecodeError):
        actual = None
    if actual != expected:
        raise ValueError("Index metadata is missing or incompatible with these embedding settings. Choose a new index_name; do not mix embedding spaces.")

**ensure_table:** Create this index if absent; reject incompatible existing embedding spaces.

In [ ]:
# Define creation of a compatible pgvector table plus retrieval indexes.
# psycopg.sql builds SQL safely: Identifier quotes table/index names, Literal quotes
# SQL values in these DDL statements, and SQL(...).format inserts those objects.
# Ordinary query values use %s parameters; placeholders cannot represent table names.
# The connection context commits on success and rolls back on an exception.
# HNSW indexes cosine-vector distance; GIN indexes the generated full-text field.
# The table comment records embedding identity, checked again on reuse.

def ensure_table(cfg):
    """Create this index if absent; reject incompatible existing embedding spaces."""
    from psycopg import sql
    name = _table_name(cfg)
    table = sql.Identifier("public", name)
    dimensions = int(cfg["embedding"]["dimensions"])
    hnsw = cfg["database"]["hnsw"]
    m, construction = int(hnsw["m"]), int(hnsw["ef_construction"])
    if dimensions < 1 or not 2 <= m <= 100 or construction < 2 * m:
        raise ValueError("Invalid vector dimension or HNSW settings (ef_construction must be >= 2*m)")
    # The psycopg connection context bounds a transaction and closes the connection on exit.
    with _connect(cfg) as connection:
        connection.execute("CREATE EXTENSION IF NOT EXISTS vector")
        exists = connection.execute("SELECT to_regclass(%s) AS name", ("public." + name,)).fetchone()["name"]
        if not exists:
            connection.execute(sql.SQL("""CREATE TABLE {} (
                chunk_id text PRIMARY KEY, doc_id text NOT NULL, source_path text NOT NULL,
                heading_path text NOT NULL, chunk_text text NOT NULL, embed_text text NOT NULL,
                token_count integer NOT NULL, fact_ids text[] NOT NULL,
                content_sha256 text NOT NULL UNIQUE, source_refs jsonb NOT NULL,
                tsv tsvector GENERATED ALWAYS AS (to_tsvector('english'::regconfig, embed_text)) STORED,
                embedding vector({}) NOT NULL
            )""").format(table, sql.Literal(dimensions)))
            metadata = json.dumps({"format_version": 1, "embedding": _embedding_signature(cfg)}, sort_keys=True)
            connection.execute(sql.SQL("COMMENT ON TABLE {} IS {}").format(table, sql.Literal(metadata)))
        _check_index_signature(connection, cfg)
        # HNSW trades exhaustive distance search for graph-based approximate nearest neighbors.
        connection.execute(sql.SQL("CREATE INDEX IF NOT EXISTS {} ON {} USING hnsw (embedding vector_cosine_ops) WITH (m = {}, ef_construction = {})").format(sql.Identifier(name + "_hnsw"), table, sql.Literal(m), sql.Literal(construction)))
        connection.execute(sql.SQL("CREATE INDEX IF NOT EXISTS {} ON {} USING gin (tsv)").format(sql.Identifier(name + "_tsv"), table))
    return name

In [ ]:
# Convert a numeric vector into pgvector's bracketed textual representation.
# all(math.isfinite(...)) rejects NaN/infinity before building the value.
# join inserts commas between numbers; later SQL passes this string as a %s value
# and casts it to vector, rather than concatenating untrusted text into SQL.

def _vector_literal(vector):
    if not vector or not all(math.isfinite(float(number)) for number in vector):
        raise ValueError("Vector must contain finite numbers")
    return "[" + ",".join(str(float(number)) for number in vector) + "]"

**ingest:** Embed first, then atomically replace stale content and update source metadata.

In [ ]:
# Define incremental ingestion: validate/embed first, then update the index atomically.
# psycopg.sql safely composes table identifiers; Jsonb adapts source-reference data
# to PostgreSQL JSONB. Content hashes decide which chunks need new embeddings.
# The first connection reads the old state; expensive embedding happens outside
# the write transaction. A later exclusive lock plus state check detects concurrent
# changes before writes. The connection context commits all writes together or
# rolls them back on error. Stale rows are deleted only after embeddings succeed;
# existing vectors are retained while IDs/provenance can be updated.

def ingest(chunks, cfg):
    """Embed first, then atomically replace stale content and update source metadata."""
    from psycopg import sql
    from psycopg.types.json import Jsonb
    chunks = _dedupe_chunks(chunks)
    if not chunks:
        raise ValueError("Refusing empty ingestion; check the corpus before removing an entire index")
    if len({chunk["chunk_id"] for chunk in chunks}) != len(chunks):
        raise ValueError("chunk_id values must be unique")
    for chunk in chunks:
        if chunk["content_sha256"] != _hash(chunk["embed_text"]):
            raise ValueError("content_sha256 must hash the exact embed_text")
        _check_chunk_limits(chunk["embed_text"], cfg)
    ensure_table(cfg)
    table = sql.Identifier("public", _table_name(cfg))
    with _connect(cfg) as connection:
        _check_index_signature(connection, cfg)
        existing = {row["content_sha256"] for row in connection.execute(sql.SQL("SELECT content_sha256 FROM {}").format(table))}
    # Hash reuse avoids recomputing embeddings for unchanged content, even if source metadata changes.
    new_chunks = [chunk for chunk in chunks if chunk["content_sha256"] not in existing]
    vectors = embed_texts([chunk["embed_text"] for chunk in new_chunks], cfg, input_type="document")
    new_vectors = {chunk["content_sha256"]: vector for chunk, vector in zip(new_chunks, vectors)}
    desired = [chunk["content_sha256"] for chunk in chunks]
    # No deletion occurs before all new embeddings and limit checks have succeeded.
    with _connect(cfg) as connection:
        # Serialize index writers; recheck state after the lock because embedding took place outside it.
        connection.execute(sql.SQL("LOCK TABLE {} IN EXCLUSIVE MODE").format(table))
        _check_index_signature(connection, cfg)
        current = {row["content_sha256"] for row in connection.execute(sql.SQL("SELECT content_sha256 FROM {}").format(table))}
        if current != existing:
            raise RuntimeError("Index changed during embedding. Retry ingestion; no rows were changed.")
        # ANY receives the desired hashes as one array-valued parameter; remove only content no longer wanted.
        connection.execute(sql.SQL("DELETE FROM {} WHERE NOT (content_sha256 = ANY(%s::text[]))").format(table), (desired,))
        # Temporary unique IDs avoid primary-key collisions when content moves between sections.
        connection.execute(sql.SQL("UPDATE {} SET chunk_id = 'pending:' || content_sha256").format(table))
        for chunk in chunks:
            # Build a positional parameter tuple in exactly the SQL column order; Jsonb adapts nested provenance.
            values = tuple(chunk[key] for key in ("chunk_id", "doc_id", "source_path", "heading_path", "chunk_text", "embed_text", "token_count", "fact_ids")) + (Jsonb(chunk["source_refs"]), chunk["content_sha256"])
            if chunk["content_sha256"] in existing:
                connection.execute(sql.SQL("UPDATE {} SET chunk_id=%s, doc_id=%s, source_path=%s, heading_path=%s, chunk_text=%s, embed_text=%s, token_count=%s, fact_ids=%s, source_refs=%s WHERE content_sha256=%s").format(table), values)
            else:
                connection.execute(sql.SQL("INSERT INTO {} (chunk_id,doc_id,source_path,heading_path,chunk_text,embed_text,token_count,fact_ids,source_refs,content_sha256,embedding) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s::vector)").format(table), values + (_vector_literal(new_vectors[chunk["content_sha256"]]),))
    counts = {"added": len(new_chunks), "unchanged": len(chunks) - len(new_chunks), "removed": len(existing - set(desired)), "total": len(chunks)}
    print("Index:", ", ".join(f"{name}={value}" for name, value in counts.items()))
    return counts

### Candidate search, optional rank fusion, and final context selection

Vector search orders cosine distance. Optional hybrid retrieval adds a full-text ranking through reciprocal rank fusion: sum 1/(k + rank). Rank fusion combines order positions, not incompatible raw scores.

**search_candidates:** Top-n cosine candidates, optionally fused with top-n full-text candidates.

In [ ]:
# Define first-stage candidate search: cosine neighbors, optionally mixed with text search.
# psycopg.sql separates quoted table names from %s-bound query/vector/count values.
# pgvector <=> is cosine distance, so 1-distance is similarity. The raw distance
# ORDER BY permits HNSW use; chunk_id breaks equal-score ties deterministically.
# Hybrid mode unions vector and lexical candidates, then uses reciprocal rank fusion:
# sum 1/(rrf_k + rank) across lists where a chunk appears. This combines ranks,
# not incomparable cosine and lexical score scales. It returns candidate metadata;
# fact_ids are carried for evaluation but are not consulted to rank candidates.

def search_candidates(question, cfg):
    """Top-n cosine candidates, optionally fused with top-n full-text candidates."""
    from psycopg import sql
    if not isinstance(question, str) or not question.strip():
        raise ValueError("question must be a nonempty string")
    vector = _vector_literal(embed_texts([question], cfg, input_type="query")[0])
    table, n = sql.Identifier("public", _table_name(cfg)), int(cfg["retrieval"]["top_n_candidates"])
    fields = "chunk_id,doc_id,source_path,heading_path,chunk_text,embed_text,token_count,fact_ids,content_sha256,source_refs"
    before = time.perf_counter()
    with _connect(cfg) as connection:
        _check_index_signature(connection, cfg)
        ef_search = int(cfg["database"]["hnsw"]["ef_search"])
        if ef_search < n:
            raise ValueError("hnsw.ef_search must be at least top_n_candidates")
        # The third argument makes ef_search transaction-local; it controls HNSW's search breadth.
        connection.execute("SELECT set_config('hnsw.ef_search', %s, true)", (str(ef_search),))
        # Keep the inner ORDER BY as the raw distance operator for HNSW.
        # WITH TIES lets the outer deterministic chunk_id order break cutoffs.
        vector_rows = connection.execute(sql.SQL(
            "SELECT " + fields + ", 1 - distance AS similarity_score FROM (SELECT " + fields +
            ", embedding <=> %s::vector AS distance FROM {} "
            "ORDER BY embedding <=> %s::vector FETCH FIRST %s ROWS WITH TIES) AS nearest "
            "ORDER BY distance, chunk_id LIMIT %s"
        ).format(table), (vector, vector, n, n)).fetchall()
        for rank, row in enumerate(vector_rows, 1):
            row.update(vector_rank=rank, lexical_rank=None, rerank_score=None, rerank_rank=None)
        rows = vector_rows
        # Optional lexical retrieval adds exact-term signals; merge by chunk ID before fusing ranks.
        if cfg["retrieval"]["hybrid"]["enabled"]:
            lexical_rows = connection.execute(sql.SQL("SELECT " + fields + ", 1 - (embedding <=> %s::vector) AS similarity_score, ts_rank_cd(tsv, plainto_tsquery('english', %s)) AS lexical_score FROM {} WHERE tsv @@ plainto_tsquery('english', %s) ORDER BY lexical_score DESC, chunk_id LIMIT %s").format(table), (vector, question, question, n)).fetchall()
            union = {row["chunk_id"]: row for row in vector_rows}
            for rank, row in enumerate(lexical_rows, 1):
                if row["chunk_id"] not in union:
                    union[row["chunk_id"]] = dict(row, vector_rank=None, rerank_score=None, rerank_rank=None)
                union[row["chunk_id"]].update(lexical_rank=rank, lexical_score=row["lexical_score"])
            k = int(cfg["retrieval"]["hybrid"]["rrf_k"])
            # A chunk present in both ranked lists receives two reciprocal-rank contributions.
            for row in union.values():
                row["rrf_score"] = sum(1 / (k + row[rank]) for rank in ("vector_rank", "lexical_rank") if row.get(rank) is not None)
            rows = sorted(union.values(), key=lambda row: (-row["rrf_score"], row["chunk_id"]))[:n]
    for rank, row in enumerate(rows, 1):
        row["candidate_rank"] = rank
    _STATS["search_seconds"] += time.perf_counter() - before
    return rows

**retrieve:** Return top-k contexts; an optional raw-logit threshold can return [].

In [ ]:
# Define the final retrieval policy used to supply contexts to the answer model.
# First search top_n candidates, then optionally rerank and apply a score threshold.
# A failed threshold returns [] (no evidence). [:top_k] limits the selected contexts;
# enumerate(..., 1) records their one-based order. A reranker threshold is invalid
# when reranking is disabled, and raw logits are not probabilities of answerability.

def retrieve(question, cfg):
    """Return top-k contexts; an optional raw-logit threshold can return []."""
    candidates = search_candidates(question, cfg)
    if cfg["rerank"]["enabled"]:
        candidates = rerank_candidates(question, candidates, cfg)
        threshold = cfg["retrieval"].get("min_rerank_score")
        if threshold is not None and (not candidates or candidates[0]["rerank_score"] < float(threshold)):
            return []
    elif cfg["retrieval"].get("min_rerank_score") is not None:
        raise ValueError("A reranker threshold cannot be used when reranking is disabled")
    result = candidates[:int(cfg["retrieval"]["top_k"])]
    for rank, row in enumerate(result, 1):
        row["retrieval_rank"] = rank
    return result

**all_chunks:** Read source/evidence metadata, without transferring stored vectors.

In [ ]:
# Define a read of stored chunk metadata without transferring large embedding vectors.
# psycopg.sql.Identifier quotes the configured schema/table; SQL.format inserts it.
# The signature check stops incompatible indexes. ORDER BY makes the inventory
# deterministic; fetchall returns dictionary rows for later inspection/count checks.

def all_chunks(cfg):
    """Read source/evidence metadata, without transferring stored vectors."""
    from psycopg import sql
    with _connect(cfg) as connection:
        _check_index_signature(connection, cfg)
        return connection.execute(sql.SQL("SELECT chunk_id,doc_id,source_path,heading_path,chunk_text,embed_text,token_count,fact_ids,content_sha256,source_refs FROM {} ORDER BY source_path,chunk_id").format(sql.Identifier("public", _table_name(cfg)))).fetchall()

### Accounting and releasing accelerator memory

These functions expose the reusable pipeline without hiding the data flow. Follow their arguments and returned records in the worked run below.

**stats:** Process-local counters: input tokens count uncached model work, excluding pad.

In [ ]:
# Return an independent snapshot of process-local retrieval counters.
# copy.deepcopy recursively copies the dictionary so later increments do not
# rewrite an earlier snapshot. Token counters measure uncached model inputs,
# excluding padding; they are not total corpus size or provider API charges.

def stats():
    """Process-local counters: input tokens count uncached model work, excluding pad."""
    return copy.deepcopy(_STATS)

**release_models:** Free BGE weights before loading a Qwen answer model; retain tokenizers/cache.

In [ ]:
# Define cleanup before another large model uses the shared memory budget.
# gc is Python's garbage collector; torch provides backend cache controls.
# Clearing _MODELS removes this pipeline's model references; gc.collect requests
# collection, and empty_cache releases unused cached accelerator allocations.
# Live tensors cannot be freed this way. Tokenizers and disk caches remain reusable.

def release_models():
    """Free BGE weights before loading a Qwen answer model; retain tokenizers/cache."""
    import gc
    import torch
    _MODELS.clear()
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Local preflight and the selected configuration

Postgres must be running and the `meridian_rag` database must exist. Setup steps for PostgreSQL and pgvector are in `docs/SETUP.md`.md`. This notebook creates only its configured index table and vector extension. It never changes roles or other databases.

Pinned model revisions make caches reproducible. Download instructions are in the guide if you move to another machine. The checks below use the local tokenizer/model files and the configured database.

In [ ]:
# Load and display the selected retrieval settings, then prepare the index table.
# load_config validates settings; an optional dictionary edit changes this in-memory
# experiment only. ensure_table is a real database operation when this cell runs:
# it creates missing vector/table/index objects and checks existing compatibility.
# Retrieval is local, but zero provider calls does not mean zero compute cost.

RAG = load_config()
# Optional in-memory override for an experiment, without changing the YAML:
# RAG["retrieval"]["top_k"] = 5
print("Index:", RAG["index_name"])
print("Embedding:", RAG["embedding"]["model_id"], RAG["embedding"]["revision"])
print("Reranker:", RAG["rerank"]["model_id"], RAG["rerank"]["revision"])
print("Provider API calls for retrieval: 0 (local inference)")
ensure_table(RAG)

## Chunk the five source documents

Inspect the complete inventory and three examples. **Heading strategy:** each `##` section is a chunk, each markdown table data row is a chunk with its header, and introductory text is retained so launch years are not lost. The brand guide has no `##` headings and stays one chunk.

`token_count` is the Qwen estimate. Exact encoder limits are checked independently. `fact_ids` labels the canonical facts supported by each chunk; one chunk can support several facts. Arc fast-charge `0` is a sentinel meaning unsupported, not a zero-minute charging time. With `corpus.fact_labels: null`, you can still build/search a future corpus, but gold-fact retrieval metrics are skipped.

In [ ]:
# Build the configured chunks and inspect their context headings and evidence labels.
# chunk_corpus reads only configured source documents and optional canonical facts.
# pd.option_context temporarily expands table display settings inside its with block.
# The nested set comprehension counts unique fact IDs across chunks; many chunks
# may reference one fact, so chunk count and fact coverage are different quantities.

chunks = chunk_corpus(RAG)
chunk_frame = pd.DataFrame(chunks)
with pd.option_context("display.max_rows", None, "display.max_colwidth", 100):
    display(chunk_frame[["chunk_id", "heading_path", "token_count", "fact_ids"]])
print("Chunks:", len(chunks), "Unique labelled facts:", len({f for c in chunks for f in c["fact_ids"]}))


In [ ]:
# Measure how two tokenizers segment the same embedded chunk text.
# The list comprehension computes actual BGE input lengths, including special tokens.
# Division aligns dataframe rows to form BGE/Qwen ratios. The overall ratio divides
# the two total token counts, rather than averaging per-chunk ratios.
# Qwen counts are an estimate for chunk sizing; BGE limits are checked separately.

chunk_frame["bge_input_tokens"] = [len(_tokenizer("embedding", RAG).encode(c["embed_text"], add_special_tokens=True)) for c in chunks]
chunk_frame["bge_to_qwen_token_ratio"] = chunk_frame["bge_input_tokens"] / chunk_frame["token_count"]
display(chunk_frame[["chunk_id", "token_count", "bge_input_tokens", "bge_to_qwen_token_ratio"]])
print("Total BGE/Qwen token ratio:", round(chunk_frame.bge_input_tokens.sum()/chunk_frame.token_count.sum(), 3))


In [ ]:
# Print three complete embedded chunks so you can inspect the actual encoder input.
# [:3] takes at most the first three list items. Embedded text may include a
# title/heading header that supplies model and section context to the retriever.

for c in chunks[:3]:
    print("\nCHUNK", c["chunk_id"], "\n", c["embed_text"])

## One real embedding, then incremental ingestion

The vector's shape is its number of dimensions; its norm should be approximately 1. The first run computes embeddings; subsequent runs reuse cached vectors and unchanged rows. `added`, `unchanged` and `removed` describe database records, not training examples. The core ingestion SQL is visible above.

In [ ]:
# Run one local document embedding and inspect its dimensions, norm and accounting.
# This is real encoder inference (or an exact cache hit) when you run the cell.
# np.asarray converts its numeric list to an array; np.linalg.norm should be near
# 1 after L2 normalization. perf_counter measures elapsed time for this call.
# Token counts use the embedding tokenizer, not the Qwen approximation.

before_embedding = stats()
t0 = time.perf_counter()
one_vector = np.asarray(embed_texts([chunks[0]["embed_text"]], RAG, input_type="document")[0])
print("Vector shape:", one_vector.shape, "Norm:", round(float(np.linalg.norm(one_vector)), 6))
print("Actual input tokens:", len(_tokenizer("embedding", RAG).encode(chunks[0]["embed_text"], add_special_tokens=True)))
print("Seconds:", round(time.perf_counter()-t0, 3))
print("Counters after embedding:", stats())

In [ ]:
# Ingest the corpus and verify that the stored chunk count matches the input.
# ingest may write the configured database and local embedding cache when run.
# all_chunks reads metadata back; assert stops if its row count differs.
# This checks storage volume, not whether retrieval finds the right evidence.

ingestion = ingest(chunks, RAG)
print("Ingestion:", ingestion)
stored_chunks = all_chunks(RAG)
assert len(stored_chunks) == len(chunks), "Stored chunk count differs from source chunks"
print("Rows in the index:", len(stored_chunks))

## Bi-encoder search versus cross-encoder reranking

**Bi-encoder:** encode question and chunk separately. Documents are precomputed; one query vector searches the index quickly. **Cross-encoder:** read the question and each candidate together to judge that particular pair. This is more expensive, so it only examines `top_n_candidates`.

`top_k` determines how many chunks reach the answer model. Reranking may improve selection but is not guaranteed to do so. Here we trace one original dev question through both rankings. Full-text hybrid fusion is off by default; when enabled, its candidate order is separately labelled.

In [ ]:
# Trace one development question through search and optional reranking.
# lab.eval_set('dev') loads dev only. The assertion checks the expected split size.
# next(...) selects the specified question ID; indexing extracts its question text.
# search_candidates encodes/searches it; rerank_candidates jointly scores each pair
# when enabled. These calls perform local model/database work if you run the cell.

dev = lab.eval_set("dev")
assert len(dev) == 36 and sum(q["answerable"] for q in dev) == 27
TRACE_ID = "q_110_charge_time_hours"
trace_question = next(q for q in dev if q["id"] == TRACE_ID)
candidates = search_candidates(trace_question["question"], RAG)
reranked = rerank_candidates(trace_question["question"], candidates, RAG) if RAG["rerank"]["enabled"] else []


In [ ]:
# Display candidate orders before and after reranking for the same dev question.
# DataFrame.reindex(columns=...) requests a stable column layout even when a list
# is empty. Vector/rerank ranks expose order changes; fact_ids let you inspect
# gold evidence coverage, but those labels were not used to choose the ranking.

print(trace_question["question"])
columns = ["chunk_id", "heading_path", "similarity_score", "vector_rank", "rerank_score", "rerank_rank", "fact_ids"]
print("First-stage candidates:")
display(pd.DataFrame(candidates).reindex(columns=columns))
print("After reranking:")
display(pd.DataFrame(reranked).reindex(columns=columns))

## Measure retrieval before measuring the answer model

For an answerable question, a hit means at least one returned chunk contains its gold fact ID. **Recall@k** is the fraction of these questions with a hit in the first k chunks. **MRR** averages `1 / first_gold_rank`; a miss contributes 0. MRR here is bounded by the candidate list, not an exhaustive corpus ranking.

We compare raw vector search and vector search plus reranking, independently of an optional hybrid experiment. Threshold filtering is evaluated as a separate effective-context decision. The 9 unanswerable questions have no gold chunk and do not enter recall or MRR.

In [ ]:
# Create an isolated evaluation configuration for vector-versus-rerank measurement.
# deepcopy prevents these experimental overrides from altering the selected RAG.
# Disabling hybrid fusion and thresholding isolates the rerank comparison.
# Four empty lists collect full traces, metric rows, top scores and missed evidence.

metric_cfg = copy.deepcopy(RAG)
metric_cfg["retrieval"]["hybrid"]["enabled"] = False
metric_cfg["retrieval"]["min_rerank_score"] = None
retrieval_records, metric_rows, score_rows, misses = [], [], [], []


In [ ]:
# Evaluate each dev query's retrieved evidence, without asking an answer model.
# Candidate lists are saved for all queries, including unknowns. Gold metrics use
# only answerable questions with canonical fact labels. enumerate(..., 1) gives
# one-based rank; next(..., None) finds the first chunk containing the target fact.
# Recall@k is a 0/1 hit; reciprocal rank is 1/rank with a miss contributing 0.
# The gold labels score a completed ranking; they never feed search/rerank inputs.

for item in dev:
    candidates = search_candidates(item["question"], metric_cfg)
    ranked = rerank_candidates(item["question"], candidates, metric_cfg) if metric_cfg["rerank"]["enabled"] else []
    retrieval_records.append({"id": item["id"], "question": item["question"], "answerable": item["answerable"],
                              "similarity": candidates, "reranked": ranked})
    score_rows.append({"id": item["id"], "answerable": item["answerable"],
                       "top_similarity": max((c["similarity_score"] for c in candidates), default=None),
                       "top_rerank": ranked[0].get("rerank_score") if ranked else None})
    if not item["answerable"] or RAG["corpus"]["fact_labels"] is None: continue
    stages = [("vector", candidates)] + ([("vector + rerank", ranked)] if metric_cfg["rerank"]["enabled"] else [])
    for stage, ordered in stages:
        first = next((rank for rank, c in enumerate(ordered, 1) if item["fact_id"] in c["fact_ids"]), None)
        metric_rows.append({"id": item["id"], "stage": stage, "recall@1": int(first is not None and first <= 1),
                            "recall@3": int(first is not None and first <= 3),
                            "recall@top_n": int(first is not None), "MRR": 1/first if first else 0})
        if first is None or first > RAG["retrieval"]["top_k"]:
            misses.append({"id": item["id"], "question": item["question"], "fact_id": item["fact_id"], "stage": stage, "first_gold_rank": first})


In [ ]:
# Average query-level evidence metrics by retrieval stage and display the misses.
# groupby('stage')[columns].mean() averages each metric over labelled answerable
# dev questions. Unanswerable questions have no gold chunk and are excluded.
# An empty metric table stays empty if this corpus has no canonical fact labels;
# missing labels are not interpreted as retrieval failures.

if metric_rows:
    retrieval_metrics = pd.DataFrame(metric_rows).groupby("stage")[["recall@1", "recall@3", "recall@top_n", "MRR"]].mean()
    display(retrieval_metrics)
    print("Every question whose gold chunk misses top_k:")
    display(pd.DataFrame(misses))
else:
    retrieval_metrics = pd.DataFrame()
    print("Gold-fact metrics skipped: corpus.fact_labels is null.")

## Can a relevance-score threshold identify unanswerable questions?

We compare the highest scores for the known and unknown questions. A perfect separating threshold on this small dev set exists only if the lowest answerable score is greater than the highest unanswerable score. Even that would be an observation on these 36 questions, not evidence that the threshold generalizes. These scores are not answerability probabilities; a document can be related to a question yet omit the requested fact.

In [ ]:
# Inspect top retrieval scores separately for answerable and unanswerable dev queries.
# to_numeric(errors='coerce') converts unavailable/nonnumeric scores to missing.
# groupby(...).agg([...]) computes count/min/mean/max per label and score type.
# These are similarity/logit distributions, not calibrated answerability probabilities.

score_frame = pd.DataFrame(score_rows)
score_frame[["top_similarity", "top_rerank"]] = score_frame[["top_similarity", "top_rerank"]].apply(pd.to_numeric, errors="coerce")
display(score_frame.groupby("answerable")[["top_similarity", "top_rerank"]].agg(["count", "min", "mean", "max"]))


In [ ]:
# Check whether any single higher-is-better threshold separates these dev groups.
# .loc[boolean_mask, column] selects known scores; ~ flips the mask for unknowns.
# dropna omits unavailable scores. Perfect separation requires min(known) >
# max(unknown); that observation on this small dev set is not production validation.
# The scatter uses y=1/0 to distinguish groups; x remains the actual model score.

threshold_observations = {}
for column in ["top_similarity", "top_rerank"]:
    known = score_frame.loc[score_frame.answerable, column].dropna()
    unknown = score_frame.loc[~score_frame.answerable, column].dropna()
    separable = bool(len(known) and len(unknown) and known.min() > unknown.max())
    threshold_observations[column] = {"perfectly_separable_on_this_dev_set": separable}
    print(column + ":", "A gap separates these dev groups; this is not a validated production threshold." if separable else "No single higher-is-better threshold perfectly separates these dev groups.")
    if len(known) and len(unknown):
        plt.figure(figsize=(8, 2.5))
        plt.scatter(known, np.ones(len(known)), label="answerable", alpha=.7)
        plt.scatter(unknown, np.zeros(len(unknown)), label="unanswerable", alpha=.7)
        plt.yticks([0,1], ["unknown", "known"]); plt.xlabel(column); plt.legend(); plt.tight_layout(); plt.show()

## Parity: visible code versus the reusable module

The following checks all 36 questions using the actual configured hybrid/rerank/threshold settings. Both paths must return identical ordered chunk IDs. Cache hits keep this cheap. An empty context after thresholding is legitimate and is represented by an empty chunk list; the answer notebook prints the configured no-evidence message.

In [ ]:
# Check that the visible notebook functions and reusable module return identical order.
# For every dev question, comprehensions extract chunk IDs from both retrieval paths.
# List equality checks values and order, including a legitimate empty context.
# all(...) requires every comparison to pass; assert stops on disagreement.
# This tests implementation parity, not factual answer quality or unseen-query recall.

parity_results = []
for item in dev:
    visible_ids = [c["chunk_id"] for c in retrieve(item["question"], RAG)]
    reusable_ids = [c["chunk_id"] for c in reusable_rag.retrieve(item["question"], RAG)]
    parity_results.append({"id": item["id"], "same_ordered_ids": visible_ids == reusable_ids})
assert all(r["same_ordered_ids"] for r in parity_results), "Visible cells and src/rag.py disagree"
print("Parity passed:", len(parity_results), "of", len(dev))

## Save the retrieval experiment

This records local model revisions, input limits, chunk count, metrics, tokens, caches and timings. Local embedding/reranking makes zero provider API calls. No monetary estimate is invented: provider prices remain null until you enter them; local hardware/electricity costs are outside this measurement.

The index is ready for notebook 12. Release both copies of the local retrieval models before loading Qwen. The saved notebook itself is delivered with empty outputs, so you can make your own observations when running it.

In [ ]:
# Assemble a reproducible retrieval report from measured results and fingerprints.
# deepcopy lets us remove internal root fields and redact the DSN in the saved copy.
# SHA-256 hashes identify config, source text and implementation versions; model
# revisions identify the encoders. Token totals, misses, metrics and cache counters
# remain distinct. None means unrecorded/not applicable, not measured zero cost.
# The dictionary comprehension fingerprints source files actually used for chunks.

report_config = copy.deepcopy(RAG)
report_config.pop("_root", None)
report_config["database"]["dsn"] = "[read from configs/rag.yaml]"
index_report = {
    "config": report_config,
    "config_sha256": hashlib.sha256((ROOT/"configs/rag.yaml").read_bytes()).hexdigest(),
    "index_name": RAG["index_name"], "chunk_count": len(chunks), "ingestion": ingestion,
    "corpus_sha256": {p: hashlib.sha256((ROOT/p).read_bytes()).hexdigest() for p in sorted({c["source_path"] for c in chunks})},
    "implementation_sha256": hashlib.sha256((ROOT/"src/rag.py").read_bytes()).hexdigest(),
    "qwen_approximate_document_tokens": int(chunk_frame.token_count.sum()),
    "bge_document_input_tokens": int(chunk_frame.bge_input_tokens.sum()),
    "embedding_model": RAG["embedding"]["model_id"], "rerank_model": RAG["rerank"]["model_id"],
    "regions": None, "execution": "local", "api_calls": 0,
    "metrics": retrieval_metrics.reset_index().to_dict("records") if not retrieval_metrics.empty else [],
    "misses": misses, "threshold_observations": threshold_observations,
    "local_measurements": stats(), "parity_measurements": reusable_rag.stats(),
    "parity_questions": len(parity_results), "elapsed_seconds": time.perf_counter()-START,
}


In [ ]:
# Write the completed report to its index-specific JSON filename when this cell runs.
# Path / joins folders; the f-string inserts the selected index name.
# json.dumps(indent=2, default=str) makes a readable record and handles Path-like
# values; write_text replaces that named report file. No provider price is invented.

INDEX_REPORT_PATH = ROOT/"runs"/f"11_rag_index_{RAG['index_name']}.json"
INDEX_REPORT_PATH.write_text(json.dumps(index_report, indent=2, default=str)+"\n")
print("Saved:", INDEX_REPORT_PATH)


In [ ]:
# Release retrieval-model weights from both the visible and reusable implementations.
# Each path has its own in-memory model cache. Freeing both makes room for Qwen
# in the answer notebook while retaining tokenizers and reusable disk caches.

release_models()
reusable_rag.release_models()